In [ ]:
!wget --no-cache -O init.py -q https://raw.githubusercontent.com/UDEA-Esp-Analitica-y-Ciencia-de-Datos/EACD-08-CLOUD/master/init.py
import init; init.init(force_download=False);
from IPython.display import Image

In [ ]:
Image("local/imgs/udea-datascience.png")


#Procesamiento basado en eventos  
## Notebook demostrativo: Productor y Consumidor con Python y `asyncio`


Aquí vamos a construir, paso a paso, una **simulación sencilla de streaming de datos** usando solo Python:

- Primero veremos un **escenario ideal sin cola**, donde el productor llama directamente al consumidor (**escenario 1**).
- Luego construiremos escenarios con **cola de mensajes** usando `asyncio.Queue` para simular:
  - Un caso **saludable**: productor y consumidor trabajan a ritmos similares (**escenario 2A**).
  - Un caso con **acumulación (backlog)** cuando el consumidor es más lento (**escenario 2B**).
  - Un caso más cercano a **batch**, donde procesamos los eventos en ventanas de tiempo (**escenario 3**).



## 1. Conceptos básicos antes de empezar

Antes de ver código, aclaremos algunas ideas que usaremos en todo el notebook.

### 🧩 ¿Qué es un *evento*?

En un sistema de streaming, un **evento** es un registro de algo que ocurrió en un momento específico. Por ejemplo:

- Una medición de temperatura de un sensor.
- Una compra en una tienda en línea.
- El clic de un usuario en una página web.

En este notebook, un evento será un **diccionario de Python** con información como:

- `sensor_id`: identificador del sensor.
- `valor`: la medición.
- `timestamp_evento`: momento en el que se generó el evento.

---

### 🧩 ¿Qué es un *productor*?

El **productor** (producer) es el componente que **genera eventos** y los envía a algún lugar:

- Puede ser un sensor físico.
- Una aplicación web que envía eventos de usuario.
- Un servicio de backend que emite eventos cuando algo sucede.

Aquí, el productor será una **función en Python** que genera eventos periódicamente.

---

### 🧩 ¿Qué es un *consumidor*?

El **consumidor** (consumer) es el componente que **recibe los eventos** y hace algo con ellos:

- Guardarlos en una base de datos.
- Enviar una alerta.
- Actualizar un dashboard en tiempo real.

Aquí, el consumidor será otra función en Python que **lee eventos** y los **procesa**.

---

### 🧩 ¿Qué es una *cola de eventos*?

En sistemas de streaming reales solemos tener una **cola** (queue) o **broker** (como Kafka, Kinesis, etc.) entre productor y consumidor.

La idea es:

- El productor **no necesita esperar** a que el consumidor termine para seguir enviando eventos.
- El consumidor procesa los eventos **a su propio ritmo**.
- Si el consumidor va más lento, la cola empieza a **acumular eventos** (backlog).

En este notebook usaremos `asyncio.Queue` para simular esa cola.

---

### ✅ Objetivo general del notebook

Al final de este notebook deberías poder responder:

- ¿Qué diferencia hay entre **procesar eventos uno a uno inmediatamente** vs **acumularlos y procesarlos después**?
- ¿Qué pasa si el productor envía eventos más rápido de lo que el consumidor puede procesar?
- ¿Por qué las colas son tan importantes en arquitecturas basadas en eventos?



## 2. Preparar el entorno en Python

En esta sección importamos las librerías que necesitamos:
- `asyncio` para manejar tareas asíncronas y la cola de eventos.
- `random` para generar valores aleatorios.
- `time` y `datetime` para trabajar con tiempos.


In [ ]:
import asyncio
import random
import time
from datetime import datetime, timezone



## 3. Escenario 1 – Productor y consumidor sin cola (llamada directa)

Primero veamos el caso **ideal y simple**:

- El productor genera un evento.
- Inmediatamente llama al consumidor para que lo procese.
- No hay cola, no hay backlog.

Visualmente sería algo así:

In [ ]:
Image("local/imgs/escenario_1.png")

En este primer escenario vamos a:

- Simular un productor que genera un evento.
- Llamar directamente a una función que representa al consumidor.
- Todo ocurre de forma **secuencial**: hasta que el consumidor no termine, el productor no genera el siguiente evento.

Esto nos sirve para entender el **comportamiento ideal y simple**, sin colas ni acumulaciones.

### 3.1. Función para generar un evento

Creamos una función que construye un **diccionario** con la información del evento.

In [ ]:
def generar_evento(sensor_id: int) -> dict:
    """
    Genera un evento simulado de un sensor.

    En un sistema real, aquí podríamos tener:
    - Lectura de un sensor físico.
    - Datos que llegan desde una API.
    - Información desde una base de datos.
    """
    valor = round(random.uniform(10, 100), 2)  # Valor aleatorio simulando una medición
    evento = {
        "sensor_id": sensor_id,
        "valor": valor,
        "timestamp_evento": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    }
    return evento

In [ ]:
# Probemos generar un evento de ejemplo
ejemplo = generar_evento(sensor_id=1)
ejemplo


### 3.2. Consumidor *sin cola* (llamada directa)

Ahora definimos un consumidor muy simple:

- Recibe un evento (diccionario).
- Simula un tiempo de procesamiento con `time.sleep(...)`.
- Imprime el evento y, si el valor es muy alto, genera una alerta.

Este consumidor es **sincrónico** y **bloqueante**, es decir, mientras está trabajando, el productor no puede avanzar.


In [ ]:
def consumidor_directo(evento: dict, tiempo_procesamiento: float = 0.8, umbral_alerta: float = 80.0):
    """
    Consumidor que procesa un evento de forma directa y sincrónica.
    """
    print(f"Consumidor directo procesando evento: {evento}")
    # Simulamos trabajo "pesado"
    time.sleep(tiempo_procesamiento)

    if evento["valor"] >= umbral_alerta:
        print(f"⚠️  ALERTA (directo) - Sensor {evento['sensor_id']} valor alto: {evento['valor']}")
    else:
        print(f"Evento procesado sin alerta (directo).")


### 3.3. Simulación del escenario 1: flujo ideal (sin cola)

En la siguiente celda:

- Generamos varios eventos dentro de un bucle.
- Por cada evento, **primero** el productor genera el evento y **luego** el consumidor lo procesa.
- Hasta que el consumidor no termina, no se genera el siguiente evento.

Observa el orden y los tiempos de ejecución.


In [ ]:
def escenario_1_sin_cola(num_eventos,
                           intervalo_entre_eventos,
                           tiempo_procesamiento):
    """
    Escenario 1: el productor y el consumidor trabajan juntos,
    sin cola intermedia. El productor espera a que el consumidor termine.
    """
    print("\n=== Escenario 1: Productor y consumidor sin cola (llamada directa) ===\n")
    sensor_id = 1
    for i in range(num_eventos):
        print(f"\n--- Iteración {i+1} ---")
        evento = generar_evento(sensor_id)
        print(f"Productor genera evento: {evento}")

        inicio = time.time()
        consumidor_directo(evento, tiempo_procesamiento)
        fin = time.time()
        duracion = fin - inicio
        print(f"Tiempo total generación + procesamiento: {duracion:.3f} segundos")

        sensor_id += 1
        # Simulamos tiempo entre eventos del productor
        time.sleep(intervalo_entre_eventos)

In [ ]:
# Ejecutamos el escenario 1
escenario_1_sin_cola(num_eventos=5,
                      intervalo_entre_eventos=0.5,
                      tiempo_procesamiento=0.8)


## 4. Escenario 2 – Productor y consumidor con cola (streaming asíncrono)

En sistemas de streaming reales casi nunca conectamos productor y consumidor directamente.

En lugar de eso, usamos una **cola**:
- El productor **envía** eventos a la cola y sigue trabajando.
- El consumidor **lee** eventos de la cola cuando puede.
- Si el consumidor va más lento, la cola empieza a **crecer** (backlog).

Visualmente:


In [ ]:
Image("local/imgs/escenario_2.png")


¿Por qué usamos `asyncio` aquí?

`asyncio` nos permite:

- Tener **tareas concurrentes** en un solo hilo.
- Crear un `asyncio.Queue` que simula una cola de eventos.
- Ejecutar en paralelo:
  - Una tarea de productor que **añade** eventos a la cola.
  - Una tarea de consumidor que **toma** eventos de la cola.

La idea es mostrar un comportamiento similar a un sistema de streaming, pero usando solo Python.



### 4.1. Productor asíncrono

Creamos un productor que:
- Genera un evento cada cierto tiempo (`intervalo_segundos`).
- Lo envía a la cola con `await queue.put(evento)`.

In [ ]:
async def productor(nombre: str, queue: asyncio.Queue, intervalo_segundos: float = 0.5):
    """
    Productor asíncrono que genera eventos periódicamente y los envía a una cola.

    :param nombre: Nombre del productor (para identificar en los logs).
    :param queue: Cola asíncrona donde se colocan los eventos.
    :param intervalo_segundos: Tiempo entre eventos simulados.
    """
    sensor_id = 1
    while True:
        evento = generar_evento(sensor_id)
        evento["productor"] = nombre
        evento["timestamp_producido"] = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

        await queue.put(evento)
        print(f"\n[{nombre}] Evento producido y enviado a la cola: {evento}")

        sensor_id += 1
        # Esperamos antes de producir el siguiente evento
        await asyncio.sleep(intervalo_segundos)


### 4.2. Consumidor asíncrono (procesamiento en streaming)

El consumidor:

- Espera a que haya un evento disponible en la cola: `evento = await queue.get()`.
- Calcula una **latencia aproximada** entre el momento en que se produjo el evento y el momento en que se procesa.
- Simula trabajo con `await asyncio.sleep(tiempo_procesamiento)`.
- Genera una alerta si el valor del evento supera un umbral.


In [ ]:
async def consumidor(nombre: str, queue: asyncio.Queue, tiempo_procesamiento: float = 0.8, umbral_alerta: float = 80.0):
    """
    Consumidor asíncrono que procesa los eventos uno a uno desde la cola.

    :param nombre: Nombre del consumidor (para logs).
    :param queue: Cola asíncrona de donde se leen los eventos.
    """
    while True:
        # Esperamos hasta que haya un evento disponible en la cola
        evento = await queue.get()
        timestamp_recibido = datetime.now(timezone.utc)

        # Cálculo de latencia (aproximado)
        try:
            t_prod = datetime.fromisoformat(evento["timestamp_producido"].replace("Z", ""))
            latencia = (timestamp_recibido - t_prod).total_seconds()
        except Exception:
            latencia = None

        print(f"[{nombre}] Evento recibido desde la cola: {evento}")
        if latencia is not None:
            print(f"[{nombre}] Latencia aproximada: {latencia:.3f} segundos")

        # Simulamos procesamiento
        await asyncio.sleep(tiempo_procesamiento)

        # Lógica de negocio simple: alerta si el valor está por encima del umbral
        if evento["valor"] >= umbral_alerta:
            print(f"[{nombre}] ⚠️  ALERTA - Sensor {evento['sensor_id']} valor alto: {evento['valor']}")
        else:
            print(f"[{nombre}] Evento procesado sin alerta.")

        # Indicamos a la cola que hemos terminado con este evento
        queue.task_done()


### 4.3. Escenario 2A – Productor y consumidor con ritmos similares (sin backlog)

En este escenario, configuramos:

- Productor: genera un evento cada **0.7 segundos**.
- Consumidor: tarda **0.5 segundos** en procesar cada evento.

Como el consumidor es ligeramente **más rápido** que el productor, la cola no debería crecer demasiado.

Observa:
- La latencia.
- El flujo continuo de eventos.


In [ ]:
async def main_escenario_2A(duracion_segundos):
    """
    Escenario 2A: productor y consumidor con ritmos similares.
    No debería acumularse una gran cantidad de eventos en la cola.
    """
    queue = asyncio.Queue()

    tarea_productor = asyncio.create_task(productor("Productor-2A", queue, intervalo_segundos=0.7))
    tarea_consumidor = asyncio.create_task(consumidor("Consumidor-2A", queue, tiempo_procesamiento=0.5))

    print("🚀 Iniciando escenario 2A (streaming sin backlog significativo)...\n")
    await asyncio.sleep(duracion_segundos)

    print("\n⏹ Deteniendo escenario 2A...")
    tarea_productor.cancel()
    tarea_consumidor.cancel()

    try:
        await tarea_productor
    except asyncio.CancelledError:
        print("[Sistema] Productor-2A detenido")

    try:
        await tarea_consumidor
    except asyncio.CancelledError:
        print("[Sistema] Consumidor-2A detenido")

    pendientes = queue.qsize()
    print(f"[Sistema] Eventos pendientes en la cola: {pendientes}")

In [ ]:
# Ejecutar el escenario 2A
await main_escenario_2A(duracion_segundos=15)


### 4.4. Escenario 2B – Consumidor más lento que el productor (backlog)

Ahora exageremos un poco:

- Productor: genera un evento cada **0.3 segundos**.
- Consumidor: tarda **1.0 segundos** en procesar cada evento.

En este caso, el productor va **más rápido** que el consumidor, así que:

- La cola empezará a **acumular eventos**.
- La latencia tenderá a **crecer**.
- Verás que, al detener la simulación, quedan eventos pendientes en la cola.

Este es un comportamiento típico cuando un sistema de streaming está **sobrecargado**.


In [ ]:
async def main_escenario_2B(duracion_segundos):
    """
    Escenario 2B: el consumidor es más lento que el productor.
    La cola empezará a acumular eventos (backlog).
    """
    queue = asyncio.Queue()

    tarea_productor = asyncio.create_task(productor("Productor-2B", queue, intervalo_segundos=0.3))
    tarea_consumidor = asyncio.create_task(consumidor("Consumidor-2B", queue, tiempo_procesamiento=1.0))

    print("🚀 Iniciando escenario 2B (streaming con posible backlog)...")
    await asyncio.sleep(duracion_segundos)

    print("\n⏹ Deteniendo escenario 2B...")
    tarea_productor.cancel()
    tarea_consumidor.cancel()

    try:
        await tarea_productor
    except asyncio.CancelledError:
        print("[Sistema] Productor-2B detenido")

    try:
        await tarea_consumidor
    except asyncio.CancelledError:
        print("[Sistema] Consumidor-2B detenido")

    pendientes = queue.qsize()
    print(f"[Sistema] Eventos pendientes en la cola: {pendientes}")

In [ ]:
# Ejecutar el escenario 2B
await main_escenario_2B(duracion_segundos=15)


## 5. Escenario 3 – Procesamiento en modo *batch* (ventanas de tiempo)

En los escenarios anteriores, el consumidor procesaba los eventos **uno a uno**, en cuanto llegaban.

Ahora vamos a cambiar el enfoque:

- El productor sigue enviando eventos a la cola.
- El consumidor **espera una ventana de tiempo** (por ejemplo, 5 segundos).
- Cada vez que termina la ventana, procesa **un lote** de eventos.

Esto se parece más a un procesamiento por **lotes (batch)**, aunque los eventos sigan llegando continuamente.


In [ ]:
Image("local/imgs/escenario_3.png")


### 5.1. Consumidor por lotes (ventanas de tiempo)

Este consumidor:

- Observa la cola durante una ventana de N segundos.
- Extrae todos los eventos que llegaron en ese periodo.
- Procesa el lote completo de una vez.


In [ ]:
async def consumidor_batch(nombre: str, queue: asyncio.Queue, ventana_segundos: float, umbral_alerta: float = 80.0):
    """
    Consumidor que procesa eventos en lotes cada cierta ventana de tiempo.
    Esto se parece más a un procesamiento por lotes (batch).
    """
    while True:
        inicio_ventana = time.time()
        lote = []

        # Durante la ventana, intentamos leer todos los eventos disponibles
        while time.time() - inicio_ventana < ventana_segundos:
            try:
                evento = queue.get_nowait()
                lote.append(evento)
                queue.task_done()
            except asyncio.QueueEmpty:
                # Si no hay eventos en este momento, esperamos un poco
                await asyncio.sleep(0.1)

        if lote:
            print(f"\n[{nombre}] Procesando lote de {len(lote)} eventos...")
            # Simulamos tiempo de procesamiento del lote
            await asyncio.sleep(2.0)

            # Lógica de negocio: buscar eventos con valor alto
            for evento in lote:
                if evento["valor"] >= umbral_alerta:
                    print(f"[{nombre}] ⚠️  ALERTA BATCH - Sensor {evento['sensor_id']} valor alto: {evento['valor']}")
                else:
                    print(f"[{nombre}] Evento procesado sin alerta - Sensor {evento['sensor_id']}")
        else:
            print(f"\n[{nombre}] No llegaron eventos en esta ventana de {ventana_segundos} s.")


### 5.2. Simulación del escenario 3

Parámetros sugeridos:

- Productor: evento cada **0.7 segundos**.
- Consumidor batch: ventana de **5 segundos**.

En cada ventana de 5 segundos veremos si llegaron eventos o no, y cuántos.


In [ ]:
async def main_escenario_3(duracion_segundos):
    """
    Escenario 3: procesamiento en modo batch con ventanas de tiempo.
    """
    queue = asyncio.Queue()

    tarea_productor = asyncio.create_task(productor("Productor-3", queue, intervalo_segundos=0.7))
    tarea_consumidor = asyncio.create_task(consumidor_batch("Consumidor-Batch", queue, ventana_segundos=5.0))

    print("🚀 Iniciando escenario 3 (batch)...")
    await asyncio.sleep(duracion_segundos)

    print("\n⏹ Deteniendo escenario 3...")
    tarea_productor.cancel()
    tarea_consumidor.cancel()

    try:
        await tarea_productor
    except asyncio.CancelledError:
        print("[Sistema] Productor-3 detenido")

    try:
        await tarea_consumidor
    except asyncio.CancelledError:
        print("[Sistema] Consumidor-Batch detenido")

In [ ]:
# Ejecutar el escenario 3
await main_escenario_3(duracion_segundos=25)


## 6. Comparación de los escenarios

Recapitulando:

1. **Escenario 1 – Sin cola (ideal y simple)**  
   - El productor espera a que el consumidor termine.  
   - No hay backlog, pero ambos están **fuertemente acoplados**.

2. **Escenario 2A – Streaming saludable**  
   - Hay cola, pero el consumidor es tan rápido (o más) que el productor.  
   - La latencia se mantiene baja.  

3. **Escenario 2B – Streaming con backlog**  
   - El consumidor es más lento que el productor.  
   - La cola empieza a crecer y la latencia aumenta.  
   - Esto refleja un sistema de streaming **sobrecargado**.

4. **Escenario 3 – Batch con ventanas**  
   - Los eventos llegan continuamente, pero se procesan en **lotes periódicos**.  
   - Se parece más a un procesamiento por lotes (**batch processing**), aunque la fuente sea continua.

## 7. Cómo relacionar este notebook con la Unidad 2 del curso

- **Procesamiento basado en eventos**: cada llamada a `generar_evento` es un evento que representa algo que pasó en el sistema.
- **Batch vs Streaming**:  
  - Streaming: escenarios 2A y 2B (procesamos evento por evento).  
  - Batch: escenario 3 (ventanas de tiempo y lotes de eventos).
- **Servicios de stream processing**: en un entorno real, la cola (`asyncio.Queue`) sería reemplazada por tecnologías como **Kafka**, **Kinesis**, **Event Hubs**, etc.
- **Fuentes y almacenamiento**: el productor representa una fuente de datos (sensores, logs, etc.) y el consumidor podría enviar los resultados a sistemas de almacenamiento, dashboards o alertas.

Este notebook es solamente una **versión didáctica y simplificada** de lo que ocurre en sistemas de streaming reales, pero permite visualizar muy bien los conceptos clave para quienes se inician en el tema.



## 8. Ejercicios propuestos

1. **Modificar frecuencias**  
   - Cambia los tiempos del productor (`intervalo_segundos`) y del consumidor (`tiempo_procesamiento`).  
   - Observa cómo cambia la latencia y el tamaño de la cola.

2. **Múltiples consumidores**  
   - Crea dos consumidores que lean de la misma cola en el escenario 2B.  
   - Comprueba si el backlog disminuye.

3. **Cambiar el umbral de alerta**  
   - Modifica `umbral_alerta` para que sea más estricto o más laxo.  
   - ¿Cuántas alertas se generan ahora?

4. **Agregar persistencia (versión conceptual)**  
   - Imagina que en lugar de solo imprimir, el consumidor guarda los eventos en una base de datos.  
   - ¿Qué implicaciones tendría un backlog muy grande?